> **Legacy exploratory notebook (June 2025, first phase of the project).** Hyper-parameters were chosen on the test files and windows were split at random, so the scores it printed are not trustworthy estimates. It is kept for the record and is audited in `scripts/audit/` and `docs/legacy_audit.html`. Outputs were cleared (they contained participant file names); set `EEGBCI_LEGACY_DATA_DIR` for the helper `utils.py`.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, precision_score, recall_score, accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
import torch.nn.functional as F
import torch.optim as optim
from glob import glob
import os
import mne
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, DepthwiseConv2D, SeparableConv2D,
                                     BatchNormalization, Activation, AveragePooling2D,
                                     Dropout, Flatten, Dense)
from tensorflow.keras.constraints import max_norm
import torch._dynamo


torch._dynamo.config.suppress_errors = True

In [ ]:
#read all file
all_files_path=glob('../data/old/*.edf')
print(len(all_files_path))
print(all_files_path)

In [ ]:
scenario = [i for i in all_files_path if "S07" not in i and "S05" not in i and "S01" not in i and "S09" not in i and "S04" not in i and "s02" not in i][:3]
print("\n".join(scenario))
print(len(scenario))

In [ ]:
scenario2 = [i for i in all_files_path if ("S01" in i or ("S04" in i and '1' not in i) ) or 's02' in i][:3]
print("\n".join(scenario2))
print(len(scenario2))

In [ ]:
from sklearn.preprocessing import MinMaxScaler
CONSIDER = 1 #s
def read_data(file_path, channels=None, _min=0, _max=1):
    datax = mne.io.read_raw_edf(file_path, preload=True, verbose=False)

    if channels:
        # Drop other channels
        existing_channels = datax.ch_names
        other_channels = [ch for ch in existing_channels if ch not in channels]
        if other_channels:
            print(f"Dropping channels: {other_channels}")
            datax.drop_channels(other_channels)
        else:
            print("No specified channels to drop were found in the data.")

    # Apply Min-Max normalization to each channel
    raw_data = datax.get_data(verbose=False)  # shape = (n_channels, n_times)

    datax._data = raw_data  # Overwrite raw data with normalized version

    # Set reference
    datax.set_eeg_reference(verbose=False)

    # Filter after normalization (optional order, depending on your use-case)
    # datax.filter(8., 30., fir_design='firwin', n_jobs=8, verbose=False)

    # Segment
    epochs = mne.make_fixed_length_epochs(datax, duration=CONSIDER, overlap=0, verbose=False)
    epochs_data = epochs.get_data(verbose=False)

    # Keep channel names
    ch_names = datax.ch_names

    return epochs_data, ch_names

In [ ]:
scenario_d = scenario + scenario2

data = [read_data(file_path)[0] for file_path in scenario_d]
_, channels = read_data(scenario[0])
n_chan = len(channels)


# Scale all data using the same scaler while keeping the shape of (batch, 30, 250)
all_data = np.concatenate(data, axis=0)

scalers = []

# 2. Fit one scaler per channel
for ch in range(n_chan):
    channel_data = all_data[:, ch, :]  # shape: (total_epochs, time)
    flat = channel_data.reshape(-1, 1)  # flatten across all epochs and timepoints
    scaler = MinMaxScaler()
    scaler.fit(flat)
    scalers.append(scaler)

# 3. Transform all data using the fitted scalers
for ch in range(n_chan):
    channel_data = all_data[:, ch, :]  # shape: (total_epochs, time)
    flat = channel_data.reshape(-1, 1)  # flatten across all epochs and timepoints
    scaled_flat = scalers[ch].transform(flat)
    all_data[:, ch, :] = scaled_flat.reshape(all_data.shape[0], -1)  # reshape back to original shape


print(all_data.shape)


# dump(scalers, 'models/scalers.joblib')

In [ ]:
tot_labels = []
for i in range(0, 240*500):
    tot_labels.append(0)
for i in range(0, 30*500):
    tot_labels.append(1)
for i in range(0, 20*500):
    tot_labels.append(0)
for i in range(0, 30*500):
    tot_labels.append(1)
for i in range(0, 20*500):
    tot_labels.append(0)
for i in range(0, 30*500):
    tot_labels.append(2)
for i in range(0, 20*500):
    tot_labels.append(0)
for i in range(0, 30*500):
    tot_labels.append(2)
for i in range(0, 60*500):
    tot_labels.append(0)

labels = []
for i in range(0, 8*60*500, CONSIDER*500):
    ### voting system
    label = tot_labels[i:i+CONSIDER*500]
    if i+CONSIDER*500 > len(tot_labels):
        break
    if label[CONSIDER*500-50:].count(1) > 40:
        labels.append(1)
    elif label[CONSIDER*500-50:].count(2) > 40:
        labels.append(2)
    elif label[CONSIDER*500-50:].count(0) > 40:
        labels.append(0)
    else:
        if label.count(1) > 0.5 * len(label):
            labels.append(1)
        elif label.count(2) > 0.5 * len(label):
            labels.append(2)
        else:
            labels.append(0)

In [ ]:
tot_labels2 = []
for i in range(0, 80*500):
    tot_labels2.append(1)
for i in range(0, 80*500):
    tot_labels2.append(2)
for i in range(0, 20*500):
    tot_labels2.append(0)
for i in range(0, 80*500):
    tot_labels2.append(1)
for i in range(0, 80*500):
    tot_labels2.append(2)
for i in range(0, 20*500):
    tot_labels2.append(0)
for i in range(0, 70*500):
    tot_labels2.append(1)
for i in range(0, 70*500):
    tot_labels2.append(2)

labels2 = []
for i in range(0, 500*500, CONSIDER*500):
    label = tot_labels2[i:i+CONSIDER*500]
    if i+CONSIDER*500 > len(tot_labels2):
        break
    if label[CONSIDER*500-50:].count(1) > 40:
        labels2.append(1)
    elif label[CONSIDER*500-50:].count(2) > 40:
        labels2.append(2)
    elif label[CONSIDER*500-50:].count(0) > 40:
        labels2.append(0)
    else:
        if label.count(1) > 0.5 * len(label):
            labels2.append(1)
        elif label.count(2) > 0.5 * len(label):
            labels2.append(2)
        else:
            labels2.append(0)

In [ ]:
print(len(labels))
print(len(labels2))

labels = labels * len(scenario)  ## TO CHAGE A CHAQUE FIUS
labels2 = labels2 * len(scenario2)  ## TO CHAGE A CHAQUE FIUS
labels = np.array(labels+labels2)

print(len(labels))

In [ ]:
data_copy = all_data.copy()
# del data

In [ ]:
print("before reg", np.unique(labels, return_counts=True))


indexes0 = np.where(labels == 0)[0]
indexes1 = np.where(labels == 1)[0]
indexes2 = np.where(labels == 2)[0]

zeros = np.unique(labels, return_counts=True)[1][0]
ones = np.unique(labels, return_counts=True)[1][1]

random_indexes0 = np.random.choice(indexes0, size=zeros-ones if zeros-ones > 0 else 0, replace=False)
random_indexes1 = np.random.choice(indexes1, size=ones-zeros if ones-zeros > 0 else 0, replace=False)
random_indexes2 = np.random.choice(indexes2, size=ones-zeros if ones-zeros > 0 else 0, replace=False)
# print(random_indexes)
labels = np.delete(labels, np.concat((random_indexes0, random_indexes1, random_indexes2)))

# print(labels)
# print(labels.shape)
y = labels
print(data_copy.shape)



# delete random_indexes from the data
# data_copy = np.delete(data_copy, np.concat((random_indexes1, random_indexes2)), axis=0)
data_copy = np.delete(data_copy, np.concat((random_indexes0, random_indexes1, random_indexes2)), axis=0)
print(np.concat((random_indexes0, random_indexes1, random_indexes2)).shape)
X = data_copy

print("after reg", np.unique(labels, return_counts=True))
print(data_copy.shape)
#count each label
print(np.unique(labels, return_counts=True))

In [ ]:
print(X.shape, y.shape)

In [ ]:
import numpy as np
from scipy.stats import skew, kurtosis

def hjorth_parameters(signal):
    """Computes Hjorth mobility and complexity for 1D signal"""
    first_deriv = np.diff(signal)
    second_deriv = np.diff(first_deriv)

    var_zero = np.var(signal)
    var_d1 = np.var(first_deriv)
    var_d2 = np.var(second_deriv)

    mobility = np.sqrt(var_d1 / var_zero) if var_zero != 0 else 0
    complexity = np.sqrt(var_d2 / var_d1) / mobility if var_d1 != 0 and mobility != 0 else 0
    return var_zero, mobility, complexity

def zero_crossings(signal):
    """Counts zero-crossings in a signal"""
    return ((signal[:-1] * signal[1:]) < 0).sum()

def extract_features(window):
    """
    Extracts features for a (30, 250) EEG window
    Returns a 1D array of shape (30 * num_features,)
    """
    num_channels = window.shape[0]   # now channels are first dim
    features = []

    for ch in range(num_channels):
        signal = window[ch, :]       # take all 250 samples of channel ch

        # mean_val = np.mean(signal)
        # std_val  = np.std(signal)
        # mediane_val = np.median(signal)
        # var_val  = np.var(signal)
        # rms_val  = np.sqrt(np.mean(signal ** 2))
        # skew_val = skew(signal)
        # kurt_val = kurtosis(signal)
        # zcr_val  = zero_crossings(signal)
        hj_activity, hj_mobility, hj_complexity = hjorth_parameters(signal)

        features.extend([
            # mediane_val
            # ,mean_val, std_val,
            # var_val, rms_val, skew_val, kurt_val,
            # zcr_val, 
            hj_activity, hj_mobility, hj_complexity
        ])

    return np.array(features)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

class FuzzyEEGClassifier:
    def __init__(self, n_rules=10, n_classes=3):
        """
        Simple Fuzzy Classifier for EEG data
        
        Args:
            n_rules: Number of fuzzy rules to generate
            n_classes: Number of output classes
        """
        self.n_rules = n_rules
        self.n_classes = n_classes
        self.rules = []
        self.feature_ranges = {}
        self.scaler = StandardScaler()
        
    def _gaussian_membership(self, x, center, sigma):
        """Calculate Gaussian membership function"""
        return np.exp(-0.5 * ((x - center) / sigma) ** 2)
    
    def _triangular_membership(self, x, a, b, c):
        """Calculate triangular membership function"""
        return np.maximum(0, np.minimum((x - a) / (b - a), (c - x) / (c - b)))
    
    def _generate_fuzzy_sets(self, X, n_sets=3):
        """Generate fuzzy sets for each feature using clustering"""
        fuzzy_sets = {}
        
        for feature_idx in range(X.shape[1]):
            feature_data = X[:, feature_idx].reshape(-1, 1)
            
            # Use KMeans to find centers for fuzzy sets
            kmeans = KMeans(n_clusters=n_sets, random_state=42)
            centers = kmeans.fit(feature_data).cluster_centers_.flatten()
            centers = np.sort(centers)
            
            # Calculate sigma for Gaussian membership functions
            if len(centers) > 1:
                sigma = np.mean(np.diff(centers)) / 2
            else:
                sigma = np.std(feature_data) / 2
            
            fuzzy_sets[feature_idx] = {
                'centers': centers,
                'sigma': sigma,
                'min': np.min(feature_data),
                'max': np.max(feature_data)
            }
            
        return fuzzy_sets
    
    def _extract_rules(self, X, y):
        """Extract fuzzy rules from training data"""
        rules = []
        
        # Generate fuzzy sets for features
        self.fuzzy_sets = self._generate_fuzzy_sets(X)
        
        # For each class, find representative patterns
        for class_label in range(self.n_classes):
            class_data = X[y == class_label]
            if len(class_data) == 0:
                continue
                
            # Use clustering to find representative patterns within each class
            n_clusters = min(self.n_rules // self.n_classes, len(class_data))
            if n_clusters > 0:
                if len(class_data) > 1:
                    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
                    cluster_centers = kmeans.fit(class_data).cluster_centers_
                else:
                    cluster_centers = class_data
                
                # Create rules based on cluster centers
                for center in cluster_centers:
                    rule = {
                        'antecedent': center,
                        'consequent': class_label,
                        'weight': 1.0
                    }
                    rules.append(rule)
        
        return rules
    
    def fit(self, X, y):
        """Train the fuzzy classifier"""
        # Normalize features
        X_scaled = self.scaler.fit_transform(X)
        
        # Extract fuzzy rules
        self.rules = self._extract_rules(X_scaled, y)
        
        print(f"Generated {len(self.rules)} fuzzy rules")
        return self
    
    def _evaluate_rule(self, x, rule):
        """Evaluate a single fuzzy rule for input x"""
        rule_strength = 1.0
        
        # Calculate membership for each feature
        for feature_idx, feature_value in enumerate(x):
            if feature_idx in self.fuzzy_sets:
                # Find closest fuzzy set center
                centers = self.fuzzy_sets[feature_idx]['centers']
                sigma = self.fuzzy_sets[feature_idx]['sigma']
                
                # Use the center closest to the rule's antecedent
                rule_center = rule['antecedent'][feature_idx]
                membership = self._gaussian_membership(feature_value, rule_center, sigma)
                rule_strength *= membership
        
        return rule_strength
    
    def predict_proba(self, X):
        """Predict class probabilities"""
        X_scaled = self.scaler.transform(X)
        probabilities = np.zeros((X_scaled.shape[0], self.n_classes))
        
        for i, x in enumerate(X_scaled):
            class_strengths = np.zeros(self.n_classes)
            
            # Evaluate all rules
            for rule in self.rules:
                rule_strength = self._evaluate_rule(x, rule)
                class_strengths[rule['consequent']] += rule_strength * rule['weight']
            
            # Normalize to get probabilities
            total_strength = np.sum(class_strengths)
            if total_strength > 0:
                probabilities[i] = class_strengths / total_strength
            else:
                # If no rules fire, assign equal probability
                probabilities[i] = np.ones(self.n_classes) / self.n_classes
        
        return probabilities
    
    def predict(self, X):
        """Predict class labels"""
        probabilities = self.predict_proba(X)
        return np.argmax(probabilities, axis=1)
    
    def get_rules_summary(self):
        """Get a summary of the fuzzy rules"""
        summary = []
        for i, rule in enumerate(self.rules):
            summary.append({
                'rule_id': i,
                'consequent_class': rule['consequent'],
                'weight': rule['weight'],
                'antecedent_sample': rule['antecedent'][:5]  # Show first 5 features
            })
        return summary

# Example usage with your EEG data
def test_fuzzy_classifier_with_eeg_features(features, labels):
    """Test the fuzzy classifier with your extracted EEG features"""
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, test_size=0.3, random_state=42, stratify=labels
    )
    
    print(f"Training set: {X_train.shape}")
    print(f"Test set: {X_test.shape}")
    print(f"Class distribution in training: {np.bincount(y_train)}")
    
    # Create and train fuzzy classifier
    fuzzy_clf = FuzzyEEGClassifier(n_rules=15, n_classes=len(np.unique(labels)))
    fuzzy_clf.fit(X_train, y_train)
    
    # Make predictions
    y_pred = fuzzy_clf.predict(X_test)
    y_pred_proba = fuzzy_clf.predict_proba(X_test)
    
    # Evaluate performance
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\nFuzzy Classifier Accuracy: {accuracy:.4f}")
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    # Show some rules
    print("\nSample Fuzzy Rules:")
    rules_summary = fuzzy_clf.get_rules_summary()
    for rule in rules_summary[:5]:  # Show first 5 rules
        print(f"Rule {rule['rule_id']}: Class {rule['consequent_class']} (weight: {rule['weight']:.2f})")
    
    return fuzzy_clf, accuracy

# Alternative: Simple fuzzy logic with membership functions
class SimpleFuzzyLogic:
    """A simpler approach using predefined membership functions"""
    
    def __init__(self):
        self.feature_stats = {}
    
    def fit(self, X, y):
        """Calculate feature statistics for membership functions"""
        self.classes = np.unique(y)
        self.feature_stats = {}
        
        for class_label in self.classes:
            class_data = X[y == class_label]
            self.feature_stats[class_label] = {
                'mean': np.mean(class_data, axis=0),
                'std': np.std(class_data, axis=0),
                'min': np.min(class_data, axis=0),
                'max': np.max(class_data, axis=0)
            }
    
    def _membership_degree(self, x, class_label):
        """Calculate membership degree for a sample to a class"""
        stats = self.feature_stats[class_label]
        
        # Simple Gaussian-like membership
        diff = x - stats['mean']
        std = stats['std'] + 1e-8  # Avoid zero division
        
        # Calculate membership as inverse of normalized distance
        membership = np.exp(-0.5 * np.sum((diff / std) ** 2))
        return membership
    
    def predict(self, X):
        """Predict using fuzzy logic"""
        predictions = []
        
        for x in X:
            memberships = {}
            for class_label in self.classes:
                memberships[class_label] = self._membership_degree(x, class_label)
            
            # Choose class with highest membership
            predicted_class = max(memberships, key=memberships.get)
            predictions.append(predicted_class)
        
        return np.array(predictions)

# Test function for your data
def run_eeg_fuzzy_analysis(features, labels):
    """Run fuzzy analysis on your EEG features"""
    
    print("=== EEG Fuzzy Classification Analysis ===")
    print(f"Features shape: {features.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"Classes: {np.unique(labels)}")
    print(f"Class distribution: {np.bincount(labels)}")
    
    # Test both approaches
    print("\n1. Testing Advanced Fuzzy Classifier:")
    fuzzy_clf, accuracy1 = test_fuzzy_classifier_with_eeg_features(features, labels)
    
    print("\n2. Testing Simple Fuzzy Logic:")
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, test_size=0.3, random_state=42, stratify=labels
    )
    
    simple_fuzzy = SimpleFuzzyLogic()
    simple_fuzzy.fit(X_train, y_train)
    y_pred_simple = simple_fuzzy.predict(X_test)
    accuracy2 = accuracy_score(y_test, y_pred_simple)
    
    print(f"Simple Fuzzy Logic Accuracy: {accuracy2:.4f}")
    print("Classification Report:")
    print(classification_report(y_test, y_pred_simple))
    
    return fuzzy_clf, simple_fuzzy

# Add this to your notebook after the feature extraction:
"""
# Run the fuzzy analysis with your extracted features
fuzzy_results = run_eeg_fuzzy_analysis(features, y)
"""

In [ ]:
# For each sample in X, extract some features.
# Example: mean, std, max, min over the last axis (axis=2)
which = ["std"]
features = []
print(X.shape)
for sample in X:
    mean_feat = np.mean(sample, axis=1)   # shape: (30,)
    std_feat = np.std(sample, axis=1)     # shape: (30,)
    med_feat = np.median(sample, axis=1)  # shape: (30,)
    var_val  = np.var(sample, axis=1)
    kurt_val = kurtosis(sample, axis=1)

    psd_feat = []
    for channel_data in sample:
        psd = np.abs(np.fft.fft(channel_data))**2  # Power spectral density
        psd_feat.append(psd[:125])  # Take the first 15 frequency bins (assuming 30 Hz sampling rate)
    # Concatenate all features for this sample
    all = []
    if "std" in which:
        all.append(std_feat)
    if "mean" in which:
        all.append(mean_feat)
    if "med" in which:
        all.append(med_feat)
    if "var" in which:
        all.append(var_val)
    if "kurt" in which:
        all.append(kurt_val)
    if "psd" in which:
        all.append(psd_feat)

    sample_features = np.concatenate(all , axis=0)
    features.append(sample_features)
features = np.array(features)  # shape: (n_samples, 30*4)
print("Extracted features shape:", features.shape)

rules = 40


In [ ]:
# Replace the last cell in your notebook with this code:
import itertools
# First, copy the FuzzyEEGClassifier class definition from the previous artifact
# Then run this:

features_c = ["mean", "std", "median", "var", "kurt", "psd"]

combinations = [combo for i in range(1, len(features_c)+1) for combo in itertools.combinations(features_c, i)]

best_features = []
best_rules = 0
best_accuracy = 0

for which in combinations:
    for rules in [10, 20, 30, 40, 50]:
        print(f"\n=== Extracting Features with {which} and {rules} rules ===")
        which = ["std"]
        features = []
        for sample in X:
            mean_feat = np.mean(sample, axis=1)   # shape: (30,)
            std_feat = np.std(sample, axis=1)     # shape: (30,)
            med_feat = np.median(sample, axis=1)  # shape: (30,)
            var_val  = np.var(sample, axis=1)
            kurt_val = kurtosis(sample, axis=1)

            psd_feat = []
            for channel_data in sample:
                psd = np.abs(np.fft.fft(channel_data))**2  # Power spectral density
                psd_feat.append(psd[:125])  # Take the first 15 frequency bins (assuming 30 Hz sampling rate)
            # Concatenate all features for this sample
            all = []
            if "std" in which:
                all.append(std_feat)
            if "mean" in which:
                all.append(mean_feat)
            if "med" in which:
                all.append(med_feat)
            if "var" in which:
                all.append(var_val)
            if "kurt" in which:
                all.append(kurt_val)
            if "psd" in which:
                all.append(psd_feat)

            sample_features = np.concatenate(all , axis=0)
            features.append(sample_features)
        features = np.array(features)  # shape: (n_samples, 30*4)
        print("Extracted features shape:", features.shape)
        if features.shape[1] <= rules:
            print(f"Skipping {which} with {rules} rules: not enough features.")
            continue

        print("=== EEG Data Summary ===")
        print(f"Features shape: {features.shape}")
        print(f"Labels shape: {y.shape}")
        print(f"Unique classes: {np.unique(y)}")
        print(f"Class distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

        # Split your data
        from sklearn.model_selection import train_test_split
        X_train, X_test, y_train, y_test = train_test_split(
            features, y, test_size=0.2, random_state=42, stratify=y
        )

        print(f"\nTraining set: {X_train.shape}, Test set: {X_test.shape}")

        # Create and train the fuzzy classifier
        print("\n=== Training Fuzzy Classifier ===")
        fuzzy_clf = FuzzyEEGClassifier(n_rules=rules, n_classes=3)
        fuzzy_clf.fit(X_train, y_train)

        # Make predictions
        print("\n=== Making Predictions ===")
        y_pred = fuzzy_clf.predict(X_test)
        y_pred_proba = fuzzy_clf.predict_proba(X_test)

        # Evaluate performance
        accuracy = accuracy_score(y_test, y_pred)
        print(f"\nFuzzy Classifier Accuracy: {accuracy:.4f}")
        if accuracy > best_accuracy:
            best_accuracy = accuracy
            best_features = which
            best_rules = rules
            print("BEST ACC!", str(accuracy))

        # print("\nDetailed Classification Report:")
        # print(classification_report(y_test, y_pred, 
        #                         target_names=['Rest', 'Task1', 'Task2']))

        # # Show confusion matrix
        # cm = confusion_matrix(y_test, y_pred)
        # print(f"\nConfusion Matrix:")
        # print(cm)

        # # Visualize some results
        # plt.figure(figsize=(12, 4))

        # # Plot 1: Confusion Matrix
        # plt.subplot(1, 3, 1)
        # plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        # plt.title('Confusion Matrix')
        # plt.colorbar()
        # tick_marks = np.arange(3)
        # plt.xticks(tick_marks, ['Rest', 'Task1', 'Task2'])
        # plt.yticks(tick_marks, ['Rest', 'Task1', 'Task2'])
        # plt.ylabel('True Label')
        # plt.xlabel('Predicted Label')

        # # Plot 2: Prediction Confidence
        # plt.subplot(1, 3, 2)
        # max_proba = np.max(y_pred_proba, axis=1)
        # plt.hist(max_proba, bins=20, alpha=0.7, color='skyblue')
        # plt.xlabel('Prediction Confidence')
        # plt.ylabel('Frequency')
        # plt.title('Prediction Confidence Distribution')

        # # Plot 3: Class-wise confidence
        # plt.subplot(1, 3, 3)
        # for class_idx in range(3):
        #     class_proba = y_pred_proba[y_test == class_idx, class_idx]
        #     plt.hist(class_proba, bins=15, alpha=0.5, 
        #             label=f'Class {class_idx}', density=True)
        # plt.xlabel('Confidence for True Class')
        # plt.ylabel('Density')
        # plt.title('Class-wise Prediction Confidence')
        # plt.legend()

        # plt.tight_layout()
        # plt.show()

        # # Show some fuzzy rules
        # print("\n=== Sample Fuzzy Rules ===")
        # rules_summary = fuzzy_clf.get_rules_summary()
        # for i, rule in enumerate(rules_summary[:10]):
        #     print(f"Rule {rule['rule_id']:2d}: If features ≈ {rule['antecedent_sample']} "
        #         f"then Class {rule['consequent_class']} (weight: {rule['weight']:.2f})")

        # # Compare with traditional classifiers
        # print("\n=== Comparison with Traditional Classifiers ===")
        # from sklearn.ensemble import RandomForestClassifier
        # from sklearn.svm import SVC
        # from sklearn.neural_network import MLPClassifier

        # classifiers = {
        #     'Fuzzy Logic': fuzzy_clf,
        #     'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
        #     'SVM': SVC(kernel='rbf', random_state=42),
        #     'Neural Network': MLPClassifier(hidden_layer_sizes=(100, 50), random_state=42, max_iter=1000)
        # }

        # results = {}
        # for name, clf in classifiers.items():
        #     if name != 'Fuzzy Logic':
        #         clf.fit(X_train, y_train)
        #         y_pred_clf = clf.predict(X_test)
        #     else:
        #         y_pred_clf = y_pred
            
        #     acc = accuracy_score(y_test, y_pred_clf)
        #     results[name] = acc
        #     print(f"{name:15s}: {acc:.4f}")

        # # Plot comparison
        # plt.figure(figsize=(10, 6))
        # names = list(results.keys())
        # accuracies = list(results.values())
        # colors = ['red', 'blue', 'green', 'orange']

        # bars = plt.bar(names, accuracies, color=colors)
        # plt.ylabel('Accuracy')
        # plt.title('Classifier Comparison on EEG Data')
        # plt.ylim(0, 1)

        # # Add value labels on bars
        # for bar, acc in zip(bars, accuracies):
        #     plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
        #             f'{acc:.3f}', ha='center', va='bottom')

        # plt.xticks(rotation=45)
        # plt.tight_layout()
        # plt.show()

        # print(f"\nBest performing classifier: {max(results, key=results.get)} "
        #     f"with accuracy: {max(results.values()):.4f}")

        # # Analyze feature importance for fuzzy rules
        # print("\n=== Feature Analysis ===")
        # print("Features used:")
        # print("- Mean values (30 channels)")
        # print("- Standard deviation (30 channels)") 
        # print("- Median values (30 channels)")
        # print("- PSD features (30 channels × 125 frequency bins)")
        # print(f"Total features: {features.shape[1]}")

        # # You can also analyze which features contribute most to the fuzzy rules
        # feature_importance = np.zeros(features.shape[1])
        # for rule in fuzzy_clf.rules:
        #     feature_importance += np.abs(rule['antecedent'])

        # feature_importance = feature_importance / len(fuzzy_clf.rules)
        # top_features = np.argsort(feature_importance)[-10:]

        # print(f"\nTop 10 most important features (indices): {top_features}")
        # print("Feature importance scores:", feature_importance[top_features])